# E3 · Primera iteración de resolución del problema
## Gestión de Tickets: clasificación automática de tickets de soporte

**Fundamentos del Procesamiento de Lenguaje Natural** · David Trueba y Gonzalo Iglesias

En E1 planteamos un sistema que, a partir del asunto y el cuerpo de un ticket, prediga a qué
departamento va (**cola**), su **prioridad** y su **tipo**. En E2 dejamos preparado el texto y
las representaciones en `data/tickets/` (`src/tickets_pipeline.ipynb`). En esta entrega entrenamos
los primeros modelos y los comparamos entre sí.

| Bloque | Qué hacemos | Requisito del enunciado |
|---|---|---|
| – | Procedimiento previo de E2: carga, limpieza, tokenización, vocabulario y embeddings | Dataset y features de E2 |
| 1 | **Detección de duplicados** | Dataset de E2 |
| 2 | Partición train / validación / test | Features de E2 |
| 3 | Métricas y registro de resultados | Accuracy, precision, recall y F1 |
| 4 | **Shallow learning** con TF-IDF: Naive Bayes, Regresión logística y SVM | 2 técnicas con representación dispersa |
| 5 | **Deep learning**: CNN y BiLSTM con atención, comparando embeddings | 2 técnicas comparando embeddings y fine-tuning |
| 6 | Prioridad y tipo con las mejores configuraciones | Las otras dos salidas de E1 |
| 7 | Tabla comparativa final y análisis de errores | Tabla comparativa |

Extras que el enunciado valora: **atención** (5.7) y **combinar features no textuales** (4.4 y 5.8).

> El notebook se puede ejecutar tanto desde `src/` como desde la raíz del proyecto y tarda unos
> 45 minutos con GPU. Con la variable de entorno `E3_FAST=1` hace una pasada rápida de prueba con
> menos datos.

## 0. Configuración

In [ ]:
import os, re, json, time, random, hashlib, warnings, urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from scipy import sparse
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             recall_score, confusion_matrix, classification_report)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

SEED = 42
FAST = os.environ.get("E3_FAST", "0") == "1"       # pasada rápida de prueba
SEEDS = [42] if FAST else [42, 43, 44]             # semillas para los modelos deep
MAX_EPOCHS = 2 if FAST else 40
N_JOBS = max(1, min(8, (os.cpu_count() or 2) - 2))

ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_DIR = ROOT / "data" / "tickets"
RAW_CSV = DATA_DIR / "raw" / "dataraw.csv"
ART_DIR = ROOT / "artifacts"
FIG_DIR = ART_DIR / "figuras"
for carpeta in (RAW_CSV.parent, ART_DIR, FIG_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

In [ ]:
# Estilo común de las gráficas: superficie clara, rejilla fina y pocos colores con un papel fijo.
COL = {"fondo": "#fcfcfb", "tinta": "#0b0b0b", "tinta2": "#52514e", "tenue": "#898781",
       "rejilla": "#e1e0d9", "eje": "#c3c2b7", "gris": "#c3c2b7",
       "azul": "#2a78d6", "naranja": "#eb6834", "aqua": "#1baf7a", "amarillo": "#eda100"}
SERIE = [COL["azul"], COL["naranja"], COL["aqua"], COL["amarillo"]]

plt.rcParams.update({
    "figure.facecolor": COL["fondo"], "axes.facecolor": COL["fondo"], "savefig.facecolor": COL["fondo"],
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial"], "font.size": 10,
    "text.color": COL["tinta"], "axes.labelcolor": COL["tinta2"],
    "xtick.color": COL["tenue"], "ytick.color": COL["tinta2"], "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.edgecolor": COL["eje"], "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": COL["rejilla"], "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.axisbelow": True, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})


def guardar(fig, nombre):
    """Guarda la figura en artifacts/figuras (la reutiliza el informe en PDF) y la muestra."""
    fig.savefig(FIG_DIR / f"{nombre}.png")
    plt.show()
    plt.close(fig)


def barras_h(ax, etiquetas, valores, color=None, fmt="{:.3f}", err=None, alto=0.55):
    """Barras horizontales finas con el valor escrito en la punta."""
    y = np.arange(len(etiquetas))
    ax.barh(y, valores, height=alto, color=COL["azul"] if color is None else color, xerr=err,
            error_kw={"ecolor": COL["tinta2"], "elinewidth": 1, "capsize": 2})
    ax.set_yticks(y, etiquetas)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.tick_params(length=0)
    tope = max(v + (err[i] if err is not None else 0) for i, v in enumerate(valores))
    for i, v in enumerate(valores):
        extra = (err[i] if err is not None else 0) + tope * 0.012
        ax.text(v + extra, i, fmt.format(v), va="center", ha="left", fontsize=9, color=COL["tinta"])
    ax.set_xlim(0, tope * 1.14)

## Procedimiento previo de la Entrega 2

In [ ]:
HF_URL = "https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets/resolve/main/"
FICHEROS = ["aa_dataset-tickets-multi-lang-5-2-50-version.csv",
            "dataset-tickets-german_normalized_50_5_2.csv",
            "dataset-tickets-multi-lang-4-20k.csv"]

if RAW_CSV.exists():
    raw = pd.read_csv(RAW_CSV)
else:
    partes = []
    for f in FICHEROS:
        ruta = RAW_CSV.parent / f
        if not ruta.exists():
            urllib.request.urlretrieve(HF_URL + f, ruta)
        partes.append(pd.read_csv(ruta))
    raw = pd.concat(partes, ignore_index=True)
    raw.to_csv(RAW_CSV, index=False)

df_e2 = raw[raw["language"] != "de"].copy()
df_e2 = df_e2[df_e2["body"].fillna("").str.strip() != ""].reset_index(drop=True)

In [ ]:
_SENT = re.compile(r"(?<=[.!?])\s+|\n+")
_TOKEN = re.compile(r"<[a-z_]+>|[^\W_]+(?:'[^\W_]+)*|[^\w\s]|_")


def arreglar_saltos(texto):
    return texto.replace("\\n", "\n").replace("’", "'")


def procesar(texto):
    oraciones = [_TOKEN.findall(o.lower()) for o in _SENT.split(texto) if o.strip()]
    return [o for o in oraciones if o]


df_e2["texto"] = [arreglar_saltos(f"{s}\n{b}").strip()
                  for s, b in zip(df_e2["subject"].fillna(""), df_e2["body"])]
df_e2["oraciones"] = df_e2["texto"].map(procesar)
df_e2["tokens"] = df_e2["oraciones"].map(lambda ors: [t for o in ors for t in o])
df_e2["oraciones_resp"] = df_e2["answer"].fillna("").map(lambda s: procesar(arreglar_saltos(s)))

In [ ]:
class Vocabulary:
    PAD, UNK = "<PAD>", "<UNK>"

    def __init__(self, docs, min_freq=2):
        self.freq = Counter(t for doc in docs for t in doc)
        self.itos = [self.PAD, self.UNK] + [t for t, n in self.freq.most_common() if n >= min_freq]
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(t, 1) for t in tokens]


def buscar_vector(kv, palabra):
    candidatos = [palabra, palabra.capitalize(), palabra.upper()]
    if palabra.endswith("'s"):
        candidatos += [palabra[:-2], palabra[:-2].capitalize()]
    for c in candidatos:
        if c in kv.key_to_index:
            return kv[c]
    return None


def matriz_desde(kv):
    rng = np.random.default_rng(SEED)
    matriz = rng.uniform(-0.25, 0.25, (len(vocab), EMB_DIM)).astype(np.float32)
    matriz[0] = 0.0
    cubierto = np.zeros(len(vocab), dtype=bool)
    for i, palabra in enumerate(vocab.itos[2:], start=2):
        v = buscar_vector(kv, palabra)
        if v is not None:
            matriz[i], cubierto[i] = v, True
    return matriz, cubierto

## 1. Datos de partida

Partimos de los tickets que deja el procedimiento previo de E2: el dataset bruto (`data/tickets/raw/dataraw.csv`, los tres CSV de
[`Tobi-Bueck/customer-support-tickets`](https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets))
sin los tickets en alemán ni los de cuerpo vacío, con el texto (`subject` + `body`) ya tokenizado.

In [ ]:
# De los dos CSV con tickets en inglés, solo el 5-2-50 trae la columna `version` informada.
df_e2["fichero"] = np.where(df_e2["version"].notna(), FICHEROS[0], FICHEROS[2])
print(f"Tickets en bruto: {len(raw):,}")
print(f"Tras quitar alemán y cuerpos vacíos (filtros de E2): {len(df_e2):,}")

### 1.1 Duplicados: una fuga de información que no vimos en E2

Al cruzar los CSV aparece un problema: dos de los ficheros son versiones distintas del mismo
dataset y **comparten tickets idénticos**. Si un ticket cae en train y su gemelo en test, el
modelo no tiene que generalizar: le basta con haberlo memorizado. Eso es una fuga de información
(*data leakage*) y hace que las métricas salgan mejores de lo que son en realidad.

In [ ]:
def clave_texto(texto):
    """Texto reducido a letras y números en minúscula: sirve para detectar tickets repetidos."""
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()


df_e2["clave"] = df_e2["texto"].map(clave_texto)

repetidos = df_e2[df_e2.duplicated("clave", keep=False)]
grupos = repetidos.groupby("clave")
print(f"Tickets con al menos un gemelo exacto: {len(repetidos):,} ({len(repetidos) / len(df_e2):.1%} del total)")
print(f"Grupos de duplicados: {grupos.ngroups:,} | aparecen en ficheros distintos: {(grupos['fichero'].nunique() > 1).mean():.0%}")
for col in ("queue", "priority", "type"):
    print(f"  grupos con '{col}' distinta entre gemelos: {(grupos[col].nunique() > 1).sum()}")

df = df_e2.drop_duplicates("clave").reset_index(drop=True)
print(f"\nTickets únicos con los que trabajamos: {len(df):,}")

Los gemelos tienen siempre las mismas etiquetas, así que no aportan información nueva: son el
mismo ticket dos veces. ¿Cuánto inflaban las métricas? Lo medimos con un clasificador rápido
(TF-IDF + SVM lineal) sobre la cola, con la partición 80/20 de E2 y con la partición limpia.

In [ ]:
def medir_fuga(datos, nombre):
    tr, te = train_test_split(datos, test_size=0.2, stratify=datos["queue"], random_state=SEED)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    clf = LinearSVC().fit(vec.fit_transform(tr["texto"]), tr["queue"])
    pred = clf.predict(vec.transform(te["texto"]))
    return {"partición": nombre, "tickets": len(datos),
            "test con gemelo en train": f"{te['clave'].isin(set(tr['clave'])).mean():.1%}",
            "accuracy": accuracy_score(te["queue"], pred),
            "f1 macro": f1_score(te["queue"], pred, average="macro")}


FUGA = pd.DataFrame([medir_fuga(df_e2, "E2 (con duplicados)"), medir_fuga(df, "E3 (sin duplicados)")])
display(FUGA.round(3))

Con duplicados, aproximadamente uno de cada cuatro tickets de test ya estaba en train y la
accuracy sube unos 9 puntos sin que el modelo sea mejor. **A partir de aquí trabajamos solo con
tickets únicos**, y por eso no reutilizamos directamente las matrices ni los splits guardados en
`data/tickets/features/` en E2, sino que recalculamos representaciones y particiones sin fuga.

### 1.2 Las tres etiquetas

El objetivo principal es la **cola** (10 departamentos). La **prioridad** (3 niveles) y el
**tipo** (4 clases) se tratan en el bloque 6.

In [ ]:
TAREAS = {"queue": sorted(df["queue"].unique()),
          "priority": ["low", "medium", "high"],
          "type": sorted(df["type"].unique())}
NOMBRE_TAREA = {"queue": "Cola", "priority": "Prioridad", "type": "Tipo"}
for tarea, clases in TAREAS.items():
    df[f"y_{tarea}"] = df[tarea].map({c: i for i, c in enumerate(clases)}).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1.5, 1, 1]})
for ax, tarea in zip(axes, TAREAS):
    conteo = df[tarea].value_counts()
    barras_h(ax, conteo.index.tolist(), conteo.values, fmt="{:,.0f}")
    ax.set_title(f"{NOMBRE_TAREA[tarea]} ({len(conteo)} clases)")
    ax.set_xlabel("tickets")
fig.tight_layout()
guardar(fig, "01_distribucion_clases")

La cola está muy desbalanceada: *Technical Support* tiene unas 20 veces más tickets que
*General Inquiry*. Por eso la métrica de referencia será el **F1 macro**, que da el mismo peso a
cada clase, y no la accuracy, que se puede subir acertando solo las clases grandes.

## 2. Tokens y partición

El preprocesado es el de E2 (procedimiento previo): saltos de línea literales convertidos en saltos
reales, división en oraciones, minúsculas y tokenización con una expresión regular que mantiene
como un único token los marcadores `<...>`, las palabras con apóstrofo (`don't`) y cada signo de
puntuación. Conservamos la puntuación en `tokens` (para las redes neuronales y las meta-features
de forma, filtrándola después al construir TF-IDF) y no se lematiza ni se quitan stopwords.

La respuesta del agente (`oraciones_resp`) **no** es una feature, porque no existe cuando llega el
ticket; como en E2, solo se usa de texto extra para entrenar el Word2Vec propio.

In [ ]:
df = df[df["tokens"].str.len() > 0].reset_index(drop=True)

print("Ejemplo:", df["tokens"].iloc[0][:28])
print(df["tokens"].str.len().describe(percentiles=[0.5, 0.95, 0.99]).round(1).to_string())

### 2.1 Partición train / validación / test

En E2 solo había train y test (80/20). Ahora hace falta un tercer conjunto, **validación**, porque
vamos a elegir hiperparámetros y a decidir cuándo parar de entrenar las redes. Si esas decisiones
se tomaran mirando test, test dejaría de ser una estimación honesta.

- **train (70 %)**: ajusta los modelos, el vocabulario, el TF-IDF y el Word2Vec propio.
- **validación (15 %)**: elige hiperparámetros y hace el *early stopping*.
- **test (15 %)**: se usa una sola vez, para la tabla final.

La partición es estratificada por cola, como en E2, y todos los modelos ven exactamente los
mismos tickets en cada conjunto, así que la comparación es justa.

In [ ]:
idx_train, idx_resto = train_test_split(df.index, test_size=0.30, stratify=df["queue"], random_state=SEED)
idx_val, idx_test = train_test_split(idx_resto, test_size=0.50, stratify=df.loc[idx_resto, "queue"],
                                     random_state=SEED)
df["split"] = "train"
df.loc[idx_val, "split"] = "val"
df.loc[idx_test, "split"] = "test"

if FAST:  # pasada rápida: submuestra de cada conjunto
    df = (df.groupby("split", group_keys=False).sample(frac=0.2, random_state=SEED)
            .reset_index(drop=True))

SPLITS = ("train", "val", "test")
IDX = {s: np.flatnonzero((df["split"] == s).to_numpy()) for s in SPLITS}
Y = {t: {s: df[f"y_{t}"].to_numpy()[IDX[s]] for s in SPLITS} for t in TAREAS}
print({s: len(i) for s, i in IDX.items()})

## 3. Métricas y registro de resultados

Para cada modelo guardamos las cuatro métricas del enunciado. Como el problema es multiclase,
precision, recall y F1 se calculan por clase y luego se promedian:

- **macro**: media simple entre clases. Una clase pequeña cuenta igual que una grande. Es la
  que usamos para precision, recall y F1, y la que decide qué modelo es mejor.
- **ponderada (weighted)**: media ponderada por el número de tickets de cada clase. La damos
  como referencia para el F1.

En la prioridad añadimos el **recall de la clase `high`**, que en E1 definimos como la crítica:
un ticket urgente clasificado como no urgente es el error más caro.

In [ ]:
RESULTADOS = []      # una fila por modelo y tarea
PREDICCIONES = {}    # (tarea, modelo) -> predicción sobre test (para matrices de confusión)


def metricas(y_true, y_pred, tarea):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    m = {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r, "f1": f1,
         "f1_ponderado": f1_score(y_true, y_pred, average="weighted")}
    if tarea == "priority":
        alta = TAREAS["priority"].index("high")
        m["recall_high"] = recall_score(y_true, y_pred, labels=[alta], average="macro", zero_division=0)
    return m


def registrar(tarea, familia, modelo, preds_test, f1_val, detalle=""):
    """Guarda las métricas de test. `preds_test` es una lista con una predicción por semilla."""
    por_semilla = pd.DataFrame([metricas(Y[tarea]["test"], p, tarea) for p in preds_test])
    fila = {"tarea": tarea, "familia": familia, "modelo": modelo, "detalle": detalle,
            "semillas": len(preds_test), "f1_val": float(np.mean(f1_val)),
            **por_semilla.mean().to_dict(),
            **{f"{k}_std": v for k, v in por_semilla.std(ddof=0).to_dict().items()}}
    RESULTADOS[:] = [r for r in RESULTADOS if not (r["tarea"] == tarea and r["modelo"] == modelo)]
    RESULTADOS.append(fila)
    PREDICCIONES[(tarea, modelo)] = preds_test[int(np.argmax(np.atleast_1d(f1_val)))]
    return fila


COLS_TABLA = ["familia", "modelo", "accuracy", "precision", "recall", "f1", "f1_ponderado"]


def tabla(tarea, extra=()):
    t = pd.DataFrame([r for r in RESULTADOS if r["tarea"] == tarea])
    cols = [c for c in [*COLS_TABLA, *extra] if c in t.columns]
    return t[cols].sort_values("f1", ascending=False).reset_index(drop=True)

## 4. Shallow learning con representación dispersa (TF-IDF)

**TF-IDF** convierte cada ticket en un vector tan largo como el vocabulario, donde casi todas
las posiciones valen cero (por eso se llama representación *dispersa*). El peso de una palabra
sube si aparece mucho en el ticket (TF) y baja si aparece en casi todos los tickets (IDF).
No tiene en cuenta el orden de las palabras, salvo lo que capturan los bigramas.

Sobre esos vectores entrenamos tres clasificadores clásicos:

| Modelo | Idea | Por qué lo probamos |
|---|---|---|
| **Naive Bayes** | Aplica Bayes suponiendo que las palabras son independientes entre sí dada la clase | Es el baseline clásico de texto: rapidísimo y difícil de sobreajustar |
| **Regresión logística** | Aprende un peso por palabra y clase, y da probabilidades con softmax | Modelo lineal discriminativo, interpretable |
| **SVM lineal** | Busca el hiperplano que separa las clases con el mayor margen | Suele ser el mejor modelo lineal en espacios dispersos de muchas dimensiones |

Como referencia mínima añadimos un clasificador trivial que siempre responde la clase más
frecuente: cualquier modelo útil tiene que superarlo.

In [ ]:
def texto_tfidf(tokens):
    """Tokens ya procesados unidos por espacios, sin signos de puntuación."""
    return " ".join(t for t in tokens if t[0].isalnum() or (len(t) > 1 and t[0] == "<"))


df["texto_tfidf"] = df["tokens"].map(texto_tfidf)
TXT = {s: df["texto_tfidf"].to_numpy()[IDX[s]] for s in SPLITS}

# Cuatro formas de construir el TF-IDF. Como en E2, se ajusta solo con train (ahora sin duplicados).
VEC_CONFIGS = {
    "unigramas": dict(ngram_range=(1, 1), stop_words=None),
    "unigramas, sin stopwords": dict(ngram_range=(1, 1), stop_words=list(ENGLISH_STOP_WORDS)),
    "uni+bigramas": dict(ngram_range=(1, 2), stop_words=None),
    "uni+bigramas, sin stopwords": dict(ngram_range=(1, 2), stop_words=list(ENGLISH_STOP_WORDS)),
}
TFIDF = {}
for nombre, cfg in VEC_CONFIGS.items():
    vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                          min_df=2, sublinear_tf=True, **cfg)
    TFIDF[nombre] = {"vec": vec, "train": vec.fit_transform(TXT["train"]),
                     "val": vec.transform(TXT["val"]), "test": vec.transform(TXT["test"])}
    print(f"{nombre:30s} -> {TFIDF[nombre]['train'].shape[1]:>6,} features")

### 4.1 Optimización de hiperparámetros

Hacemos una **búsqueda en rejilla** (*grid search*): se prueban todas las combinaciones de una
lista de valores, se entrena cada una con train y se mide su **F1 macro en validación**. Gana la
que más F1 macro saca. Test no interviene.

Hiperparámetros que se exploran:

- **Del TF-IDF** (4 opciones): solo unigramas o unigramas + bigramas, y quitar o no las stopwords.
- **Naive Bayes**: el suavizado `alpha` (evita probabilidad cero para palabras no vistas en una
  clase) y la variante, *Multinomial* o *Complement* (pensada para clases desbalanceadas).
- **Regresión logística y SVM**: `C`, la inversa de la regularización (C alto deja al modelo
  ajustarse más a train), y `class_weight`, que con `balanced` penaliza más los errores en las
  clases pequeñas.

In [ ]:
GRIDS = {
    "Naive Bayes": [{"variante": v, "alpha": a} for v in ("multinomial", "complement")
                    for a in (0.01, 0.05, 0.1, 0.3, 1.0)],
    "Regresión logística": [{"C": c, "class_weight": w} for c in (1, 10, 100) for w in (None, "balanced")],
    "SVM lineal": [{"C": c, "class_weight": w} for c in (0.03, 0.1, 0.3, 1, 3) for w in (None, "balanced")],
}
if FAST:
    GRIDS = {k: v[:2] for k, v in GRIDS.items()}


def crear_shallow(nombre, p):
    if nombre == "Naive Bayes":
        return (ComplementNB if p["variante"] == "complement" else MultinomialNB)(alpha=p["alpha"])
    if nombre == "Regresión logística":
        return LogisticRegression(C=p["C"], class_weight=p["class_weight"], max_iter=1000)
    return LinearSVC(C=p["C"], class_weight=p["class_weight"])


def _probar(nombre, p, vec_nombre, Xtr, ytr, Xva, yva):
    pred = crear_shallow(nombre, p).fit(Xtr, ytr).predict(Xva)
    return {"modelo": nombre, "tfidf": vec_nombre, **p,
            "f1_val": f1_score(yva, pred, average="macro"), "acc_val": accuracy_score(yva, pred)}


def busqueda_shallow(tarea):
    """Grid search sobre validación. Devuelve todas las combinaciones probadas."""
    trabajos = [delayed(_probar)(nombre, p, v, TFIDF[v]["train"], Y[tarea]["train"],
                                 TFIDF[v]["val"], Y[tarea]["val"])
                for nombre, grid in GRIDS.items() for p in grid for v in VEC_CONFIGS]
    return pd.DataFrame(Parallel(n_jobs=N_JOBS)(trabajos))


def evaluar_shallow(tarea, grid):
    """Reentrena la mejor combinación de cada modelo y la evalúa en test."""
    mejores = {}
    pred = DummyClassifier(strategy="most_frequent").fit(TFIDF["unigramas"]["train"], Y[tarea]["train"])
    registrar(tarea, "Baseline", "Clase mayoritaria",
              [pred.predict(TFIDF["unigramas"]["test"])],
              f1_score(Y[tarea]["val"], pred.predict(TFIDF["unigramas"]["val"]), average="macro"))
    for nombre in GRIDS:
        mejor = grid[grid["modelo"] == nombre].sort_values("f1_val", ascending=False).iloc[0]
        p = {k: (None if pd.isna(mejor[k]) else mejor[k]) for k in GRIDS[nombre][0]}
        p = {k: (v.item() if hasattr(v, "item") else v) for k, v in p.items()}   # tipos nativos de Python
        modelo = crear_shallow(nombre, p).fit(TFIDF[mejor["tfidf"]]["train"], Y[tarea]["train"])
        detalle = f"{mejor['tfidf']}; " + ", ".join(f"{k}={v}" for k, v in p.items())
        registrar(tarea, "Shallow", f"TF-IDF + {nombre}",
                  [modelo.predict(TFIDF[mejor["tfidf"]]["test"])], mejor["f1_val"], detalle)
        mejores[nombre] = {"modelo": modelo, "tfidf": mejor["tfidf"], "params": p, "f1_val": mejor["f1_val"]}
    return mejores


t0 = time.time()
GRID_SHALLOW = {"queue": busqueda_shallow("queue")}
MEJOR_SHALLOW = {"queue": evaluar_shallow("queue", GRID_SHALLOW["queue"])}
print(f"{len(GRID_SHALLOW['queue'])} combinaciones probadas en {time.time() - t0:.0f} s")
for nombre, m in MEJOR_SHALLOW["queue"].items():
    print(f"  {nombre:20s} F1 val = {m['f1_val']:.3f} | {m['tfidf']} | {m['params']}")

In [ ]:
# Cómo cambia el F1 de validación con la regularización C, según el TF-IDF usado.
g = GRID_SHALLOW["queue"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, nombre in zip(axes, ("Regresión logística", "SVM lineal")):
    sub = g[g["modelo"] == nombre]
    for color, v in zip(SERIE, VEC_CONFIGS):
        curva = sub[sub["tfidf"] == v].groupby("C")["f1_val"].max()   # mejor class_weight para cada C
        ax.plot(curva.index, curva.values, color=color, lw=2, marker="o", ms=6,
                markeredgecolor=COL["fondo"], markeredgewidth=1.5, label=v)
    ax.set_xscale("log")
    ax.set_title(nombre)
    ax.set_xlabel("C (más alto = menos regularización)")
axes[0].set_ylabel("F1 macro en validación")
axes[1].legend(title="TF-IDF", loc="lower right")
fig.tight_layout()
guardar(fig, "02_grid_shallow")

In [ ]:
display(tabla("queue").round(3))

### 4.2 Combinar el texto con features no textuales

El dataset solo trae texto y etiquetas, así que las features no textuales las calculamos
nosotros a partir de la **forma** del ticket, no de sus palabras: longitud, número de oraciones,
si tiene asunto, signos de interrogación y exclamación, cifras, marcadores anonimizados
(`<tel_num>`, `<acc_num>`...), proporción de mayúsculas y saltos de línea.

No usamos como entrada `type`, `priority` ni los `tag_*`: son etiquetas que pone el agente
después de leer el ticket, y usarlas sería otra fuga de información.

In [ ]:
def meta_features(fila):
    texto, tokens = fila["texto"], fila["tokens"]
    asunto = fila["subject"] if isinstance(fila["subject"], str) else ""
    letras = [c for c in texto if c.isalpha()]
    return {
        "n_tokens": len(tokens),
        "n_caracteres": len(texto),
        "n_oraciones": len(fila["oraciones"]),
        "long_media_palabra": float(np.mean([len(t) for t in tokens if t[0].isalnum()] or [0])),
        "tiene_asunto": float(bool(asunto.strip())),
        "n_tokens_asunto": len(_TOKEN.findall(asunto.lower())),
        "n_interrogaciones": tokens.count("?"),
        "n_exclamaciones": tokens.count("!"),
        "n_cifras": sum(t[0].isdigit() for t in tokens),
        "n_marcadores": sum(len(t) > 1 and t[0] == "<" for t in tokens),
        "ratio_mayusculas": sum(c.isupper() for c in letras) / max(1, len(letras)),
        "n_saltos_linea": texto.count("\n"),
    }


META = pd.DataFrame([meta_features(f) for f in df.to_dict("records")])
CONTEOS = [c for c in META.columns if c.startswith("n_")]
META[CONTEOS] = np.log1p(META[CONTEOS])                  # los conteos tienen cola larga
escalador = StandardScaler().fit(META.iloc[IDX["train"]])     # se ajusta solo con train
M = {s: escalador.transform(META.iloc[IDX[s]]).astype(np.float32) for s in SPLITS}
N_META = M["train"].shape[1]
print(N_META, "meta-features:", list(META.columns))

Tres experimentos sobre la cola: solo meta-features (¿llevan señal por sí solas?), y el mejor
modelo lineal con TF-IDF + meta-features. Como el TF-IDF está normalizado y las meta-features
no, se multiplican por un peso `w` que también se elige en validación.

In [ ]:
def con_meta(X, meta, w):
    return sparse.hstack([X, sparse.csr_matrix(meta * w)]).tocsr()


solo_meta = LogisticRegression(max_iter=2000, class_weight="balanced").fit(M["train"], Y["queue"]["train"])
registrar("queue", "Shallow", "Solo meta-features + Regresión logística",
          [solo_meta.predict(M["test"])],
          f1_score(Y["queue"]["val"], solo_meta.predict(M["val"]), average="macro"),
          f"{N_META} features de forma, sin texto")

mejor_lineal = max(("Regresión logística", "SVM lineal"), key=lambda n: MEJOR_SHALLOW["queue"][n]["f1_val"])
base = MEJOR_SHALLOW["queue"][mejor_lineal]
X = TFIDF[base["tfidf"]]
pruebas = []
for w in (0.03, 0.1, 0.3, 1.0):
    m = crear_shallow(mejor_lineal, base["params"]).fit(con_meta(X["train"], M["train"], w), Y["queue"]["train"])
    f1v = f1_score(Y["queue"]["val"], m.predict(con_meta(X["val"], M["val"], w)), average="macro")
    pruebas.append((f1v, w, m))
    print(f"  w = {w:<5} F1 val = {f1v:.4f}   (solo texto: {base['f1_val']:.4f})")
f1v, w, m = max(pruebas, key=lambda t: t[0])
registrar("queue", "Shallow", f"TF-IDF + meta-features + {mejor_lineal}",
          [m.predict(con_meta(X["test"], M["test"], w))], f1v, f"{base['tfidf']}; w={w}")
display(tabla("queue").round(3))

## 5. Deep learning: CNN y BiLSTM con atención

Los modelos anteriores tratan el ticket como una bolsa de palabras. Las redes de este bloque
leen el ticket como una **secuencia**: cada token se sustituye por un vector denso de 300
dimensiones (*embedding*) y la red aprende a combinar esos vectores teniendo en cuenta el orden.

### 5.1 Vocabulario y secuencias

Usamos la clase `Vocabulary` del procedimiento previo (la de E2, adaptada con `<PAD>` (índice 0) para el relleno de
secuencias): se construye solo con train, reserva `<UNK>` para las palabras desconocidas y descarta
las que aparecen menos de 2 veces. Cada ticket pasa a ser una lista de índices, recortada o
rellenada con `<PAD>` hasta 200 tokens (el 99 % de los tickets tiene menos de 180).

In [ ]:
_tokens = df["tokens"].tolist()
TOKENS = {s: [_tokens[i] for i in IDX[s]] for s in SPLITS}
vocab = Vocabulary(TOKENS["train"], min_freq=2)
MAX_LEN, EMB_DIM = 200, 300


def a_matriz(docs):
    X = np.zeros((len(docs), MAX_LEN), dtype=np.int64)
    L = np.zeros(len(docs), dtype=np.int64)
    for i, doc in enumerate(docs):
        ids = vocab.encode(doc)[:MAX_LEN]
        X[i, :len(ids)], L[i] = ids, len(ids)
    return X, L


DATOS = {}
for s in SPLITS:
    X_ids, L_ids = a_matriz(TOKENS[s])
    DATOS[s] = {"X": torch.tensor(X_ids, device=DEVICE), "L": torch.tensor(L_ids),
                "M": torch.tensor(M[s], device=DEVICE)}
YT = {t: {s: torch.tensor(Y[t][s], device=DEVICE) for s in SPLITS} for t in TAREAS}
print(f"Vocabulario: {len(vocab):,} tokens | tipos distintos en train: {len(vocab.freq):,}")

### 5.2 Tres formas de inicializar los embeddings

El enunciado pide comparar tipos de embeddings y el efecto del *fine-tuning*. Probamos cinco
variantes, que salen de combinar tres orígenes con dos formas de usarlos:

| Origen | Qué es |
|---|---|
| **Aleatorio** | Vectores al azar que la red aprende desde cero, solo con nuestros tickets |
| **Google News** | Word2Vec preentrenado con unos 100.000 millones de palabras de noticias. Conoce el inglés general pero no nuestro dominio |
| **Word2Vec propio** | Mismo esquema que el `w2v_all` de E2 (tickets y respuestas), reentrenado solo sobre el `train` ya sin duplicados. Poco texto, pero del dominio |

- **Congelado**: los vectores no cambian durante el entrenamiento. La red solo aprende a usarlos.
- **Fine-tuning**: los vectores también se actualizan, así que pueden adaptarse a la tarea. Hay
  más parámetros que aprender y, con ello, más riesgo de sobreajuste.

Para Google News mantenemos los *fallbacks* de E2 (`buscar_vector`, en el procedimiento previo): si la palabra en minúscula no está, se prueba
con mayúscula inicial, en mayúsculas y sin el posesivo `'s`. Las palabras sin vector reciben uno
aleatorio.

In [ ]:
# Google News: 1,6 GB de descarga y 3,6 GB en RAM. Solo se carga una vez: nos quedamos con las
# filas de nuestro vocabulario y las guardamos en disco.
huella = hashlib.md5(" ".join(vocab.itos).encode("utf-8")).hexdigest()[:10]
cache = ART_DIR / f"emb_google_news_{huella}.npz"
if cache.exists():
    z = np.load(cache)
    E_GOOGLE, CUB_GOOGLE = z["matriz"], z["cubierto"]
else:
    import gensim.downloader as api
    from gensim.models import KeyedVectors
    ruta_w2v = api.load("word2vec-google-news-300", return_path=True)
    kv = KeyedVectors.load_word2vec_format(ruta_w2v, binary=True, limit=300_000 if FAST else None)
    E_GOOGLE, CUB_GOOGLE = matriz_desde(kv)
    np.savez_compressed(cache, matriz=E_GOOGLE, cubierto=CUB_GOOGLE)
    del kv

# Word2Vec propio: tickets + respuestas, solo de train (para que test no influya en los vectores).
from gensim.models import Word2Vec

_ors, _ors_resp = df["oraciones"].tolist(), df["oraciones_resp"].tolist()
corpus_w2v = [o for i in IDX["train"] for o in _ors[i] + _ors_resp[i]]
w2v = Word2Vec(corpus_w2v, vector_size=EMB_DIM, window=5, min_count=2, sg=1, negative=10,
               epochs=10, workers=4, seed=SEED)
E_PROPIO, CUB_PROPIO = matriz_desde(w2v.wv)
print(f"Word2Vec propio: {len(corpus_w2v):,} oraciones, {sum(map(len, corpus_w2v)):,} tokens, "
      f"{len(w2v.wv):,} palabras")


def cobertura(nombre, cubierto):
    fila = {"embedding": nombre, "tipos del vocabulario con vector": cubierto[2:].mean()}
    for s in ("val", "test"):
        ids = DATOS[s]["X"].cpu().numpy()
        fila[f"tokens de {s} con vector"] = cubierto[ids[ids > 0]].mean()
    return fila


COBERTURA = pd.DataFrame([cobertura("Google News", CUB_GOOGLE), cobertura("Word2Vec propio", CUB_PROPIO)])
display(COBERTURA.set_index("embedding").style.format("{:.2%}"))

EMBEDDINGS = {   # nombre -> (matriz inicial, ¿congelado?)
    "Aleatorio": (None, False),
    "Google News congelado": (E_GOOGLE, True),
    "Google News fine-tuning": (E_GOOGLE, False),
    "Word2Vec propio congelado": (E_PROPIO, True),
    "Word2Vec propio fine-tuning": (E_PROPIO, False),
}

### 5.3 Las arquitecturas (las vistas en clase, Unidad 3)

**CNN para texto (Kim, 2014).** Filtros de convolución de anchura 3, 4 y 5 recorren el ticket y
cada uno se activa cuando ve un patrón concreto de 3, 4 o 5 palabras seguidas (un n-grama
aprendido). De cada filtro nos quedamos solo con su activación máxima (*max-over-time pooling*):
importa si el patrón aparece, no dónde. Las activaciones se concatenan y pasan por una capa
densa con dropout y softmax. En el artículo original, las variantes que comparamos aquí se
llaman *CNN-rand* (embeddings aleatorios), *CNN-static* (congelados) y *CNN-non-static*
(fine-tuning).

**RNN bidireccional (LSTM o GRU).** La red lee el ticket palabra a palabra manteniendo una
memoria; al ser bidireccional, otra lo lee de derecha a izquierda, de modo que cada palabra
queda representada con su contexto por ambos lados. LSTM y GRU usan *puertas* para decidir qué
recordar y qué olvidar, y así evitan el desvanecimiento del gradiente de la RNN básica. La GRU
tiene menos puertas y menos pesos; según la regla vista en clase suele ir mejor con pocos datos,
así que el tipo de celda se trata como un hiperparámetro más (5.5). El relleno se gestiona con
*packed sequences*, para que la red no procese los `<PAD>`.

**Atención.** Sin atención, el ticket entero se resume en el último estado oculto, que hace de
cuello de botella. Con atención la red vuelve a mirar todos los estados. Se sigue el cálculo de
clase, con una adaptación: al clasificar no hay decoder, así que el «estado actual» $q$ es el
estado final de la RNN.

1. Puntuación de similitud de cada estado oculto $h_t$ con el estado actual: $s_t = h_t^\top q$ (producto escalar).
2. Softmax sobre las puntuaciones para obtener los pesos de atención: $\alpha_t = e^{s_t} / \sum_k e^{s_k}$.
3. Suma de los estados ponderada por su peso, el vector de contexto: $c = \sum_t \alpha_t h_t$.
4. Se concatena el contexto con el estado actual, $[c\,;q]$, y eso entra en la capa de salida.

En 5.7 se compara además con la variante *general* de Luong et al. (2015), $s_t = h_t^\top W q$,
que aprende una matriz $W$ para decidir qué buscar.

Las dos redes admiten una entrada opcional de meta-features, que se concatena a la
representación del texto justo antes de la capa de salida (5.8).

In [ ]:
def capa_embedding(matriz, congelar):
    if matriz is None:
        emb = nn.Embedding(len(vocab), EMB_DIM, padding_idx=0)
        nn.init.uniform_(emb.weight, -0.25, 0.25)
        with torch.no_grad():
            emb.weight[0].zero_()
        return emb
    return nn.Embedding.from_pretrained(torch.tensor(matriz), freeze=congelar, padding_idx=0)


class RedTexto(nn.Module):
    """Parte común: capa de embedding, rama de meta-features y capa de salida."""

    def __init__(self, matriz, congelar, dim_texto, n_clases, dropout, n_meta):
        super().__init__()
        self.emb = capa_embedding(matriz, congelar)
        self.dropout = nn.Dropout(dropout)
        self.meta = nn.Sequential(nn.Linear(n_meta, 16), nn.ReLU()) if n_meta else None
        self.salida = nn.Linear(dim_texto + (16 if n_meta else 0), n_clases)
        self.atencion = None          # aquí se guardan los últimos pesos de atención calculados

    def clasificar(self, rep, meta):
        rep = self.dropout(rep)
        if self.meta is not None:
            rep = torch.cat([rep, self.meta(meta)], dim=1)
        return self.salida(rep)


class TextCNN(RedTexto):
    def __init__(self, matriz, congelar, n_clases, n_filtros=100, tamanos=(3, 4, 5), dropout=0.5, n_meta=0):
        super().__init__(matriz, congelar, n_filtros * len(tamanos), n_clases, dropout, n_meta)
        self.convs = nn.ModuleList([nn.Conv1d(EMB_DIM, n_filtros, k) for k in tamanos])

    def forward(self, x, longitudes, meta=None):
        e = self.emb(x).transpose(1, 2)                                              # (lote, 300, tokens)
        rep = torch.cat([F.relu(conv(e)).amax(dim=2) for conv in self.convs], dim=1)   # max-over-time
        return self.clasificar(rep, meta)


class BiRNN(RedTexto):
    """LSTM o GRU bidireccional. `atencion` puede ser "no", "producto escalar" o "general"."""

    def __init__(self, matriz, congelar, n_clases, celda="LSTM", oculto=128, dropout=0.5,
                 atencion="producto escalar", n_meta=0):
        super().__init__(matriz, congelar, (2 if atencion == "no" else 4) * oculto, n_clases, dropout, n_meta)
        self.rnn = getattr(nn, celda)(EMB_DIM, oculto, batch_first=True, bidirectional=True)
        self.tipo_atencion = atencion
        self.proyeccion = nn.Linear(2 * oculto, 2 * oculto, bias=False) if atencion == "general" else None

    def forward(self, x, longitudes, meta=None):
        paquete = pack_padded_sequence(self.emb(x), longitudes, batch_first=True, enforce_sorted=False)
        salida, h = self.rnn(paquete)
        h = h[0] if isinstance(h, tuple) else h                # la LSTM devuelve (h, c); la GRU, solo h
        estado = torch.cat([h[0], h[1]], dim=1)                 # estado final de cada dirección
        if self.tipo_atencion == "no":
            return self.clasificar(estado, meta)
        H, _ = pad_packed_sequence(salida, batch_first=True, total_length=x.size(1))    # un estado por token
        consulta = estado if self.proyeccion is None else self.proyeccion(estado)
        puntos = torch.bmm(H, consulta.unsqueeze(2)).squeeze(2)                        # 1) similitud
        alfa = torch.softmax(puntos.masked_fill(x == 0, float("-inf")), dim=1)         # 2) pesos (sin el relleno)
        self.atencion = alfa.detach()
        contexto = torch.bmm(alfa.unsqueeze(1), H).squeeze(1)                          # 3) suma ponderada
        return self.clasificar(torch.cat([contexto, estado], dim=1), meta)             # 4) contexto + estado


def crear_red(arq, matriz, congelar, n_clases, hp, n_meta=0):
    if arq == "CNN":
        return TextCNN(matriz, congelar, n_clases, n_filtros=hp["n_filtros"], dropout=hp["dropout"], n_meta=n_meta)
    return BiRNN(matriz, congelar, n_clases, celda=hp["celda"], oculto=hp["oculto"], dropout=hp["dropout"],
                 atencion=hp["atencion"], n_meta=n_meta)


def nombre_arq(arq, hp):
    if arq == "CNN":
        return "CNN"
    sufijo = {"no": "sin atención", "producto escalar": "+ atención", "general": "+ atención general"}
    return f"Bi{hp['celda']} {sufijo[hp['atencion']]}"

### 5.4 Entrenamiento

Todas las redes se entrenan igual, para que lo único que cambie entre experimentos sea lo que
queremos comparar:

- Función de pérdida: entropía cruzada categórica. Optimizador **Adam**, lotes de 64 tickets.
- ***Early stopping***: tras cada época se mide el F1 macro en validación; si pasan 5 épocas
  sin mejorar se para y se recupera el mejor modelo. El máximo son 40 épocas.
- **Dropout** antes de la capa de salida.
- Cada lote junta tickets de longitud parecida, para rellenar menos y entrenar más rápido.
- Cada configuración final se entrena con **3 semillas** y se da la media y la desviación
  típica. Una red cambia de un entrenamiento a otro por la inicialización aleatoria, así que
  una diferencia más pequeña que esa desviación no significa nada.

In [ ]:
LR, TAM_LOTE, PACIENCIA = 1e-3, 64, 5
PESOS = {t: compute_class_weight("balanced", classes=np.arange(len(c)), y=Y[t]["train"])
         for t, c in TAREAS.items()}


def lote(split, idx, usar_meta):
    """Lote recortado a la longitud de su ticket más largo. `idx` son posiciones (en CPU)."""
    d, en_gpu = DATOS[split], idx.to(DEVICE)
    longitudes = d["L"][idx]
    ancho = max(int(longitudes.max()), 5)            # 5 = filtro más ancho de la CNN
    return d["X"][en_gpu][:, :ancho], longitudes, (d["M"][en_gpu] if usar_meta else None), en_gpu


def lotes_de_train(gen):
    """Baraja train y junta en cada lote tickets de longitud parecida."""
    L = DATOS["train"]["L"]
    orden, trozo, lotes = torch.randperm(len(L), generator=gen), TAM_LOTE * 50, []
    for i in range(0, len(orden), trozo):
        bloque = orden[i:i + trozo]
        bloque = bloque[torch.argsort(L[bloque])]
        lotes += [bloque[j:j + TAM_LOTE] for j in range(0, len(bloque), TAM_LOTE)]
    return [lotes[k] for k in torch.randperm(len(lotes), generator=gen)]


@torch.no_grad()
def predecir(red, split, usar_meta=False, con_atencion=False):
    red.eval()
    n, logits, atenciones = len(DATOS[split]["L"]), [], []
    for i in range(0, n, 512):
        x, longitudes, meta, _ = lote(split, torch.arange(i, min(i + 512, n)), usar_meta)
        logits.append(red(x, longitudes, meta))
        if con_atencion:
            atenciones.append(F.pad(red.atencion, (0, MAX_LEN - x.size(1))))
    logits = torch.cat(logits)
    return (logits, torch.cat(atenciones)) if con_atencion else logits


def entrenar(red, tarea, *, semilla, pesos_clase, usar_meta=False):
    opt = torch.optim.Adam([p for p in red.parameters() if p.requires_grad], lr=LR)
    w = torch.tensor(PESOS[tarea], dtype=torch.float32, device=DEVICE) if pesos_clase else None
    perdida = nn.CrossEntropyLoss(weight=w)
    gen = torch.Generator().manual_seed(semilla)
    n, mejor, historia = len(DATOS["train"]["L"]), {"f1": -1.0, "epoca": 0}, []
    for epoca in range(1, MAX_EPOCHS + 1):
        red.train()
        suma = 0.0
        for idx in lotes_de_train(gen):
            x, longitudes, meta, en_gpu = lote("train", idx, usar_meta)
            opt.zero_grad()
            loss = perdida(red(x, longitudes, meta), YT[tarea]["train"][en_gpu])
            loss.backward()
            nn.utils.clip_grad_norm_(red.parameters(), 5.0)
            opt.step()
            suma += loss.item() * len(idx)
        logits = predecir(red, "val", usar_meta)
        f1 = f1_score(Y[tarea]["val"], logits.argmax(1).cpu().numpy(), average="macro")
        historia.append({"epoca": epoca, "loss_train": suma / n,
                         "loss_val": perdida(logits, YT[tarea]["val"]).item(), "f1_val": f1})
        if f1 > mejor["f1"]:
            mejor = {"f1": f1, "epoca": epoca,
                     "estado": {k: v.detach().clone() for k, v in red.state_dict().items()}}
        elif epoca - mejor["epoca"] >= PACIENCIA:
            break
    red.load_state_dict(mejor["estado"])
    return mejor["f1"], mejor["epoca"], pd.DataFrame(historia)


HISTORIAS, REDES, DEEP_LOG = {}, {}, []


def experimento(tarea, arq, emb, hp, *, usar_meta=False, semillas=SEEDS, apuntar=True):
    """Entrena una configuración con varias semillas. Con `apuntar` la evalúa en test y la registra."""
    matriz, congelar = EMBEDDINGS[emb]
    nombre = f"{nombre_arq(arq, hp)}{' + meta-features' if usar_meta else ''} · {emb}"
    f1s, epocas, preds, t0 = [], [], [], time.time()
    for s in semillas:
        set_seed(s)
        red = crear_red(arq, matriz, congelar, len(TAREAS[tarea]), hp, N_META if usar_meta else 0).to(DEVICE)
        f1, epoca, historia = entrenar(red, tarea, semilla=s, pesos_clase=hp["pesos_clase"], usar_meta=usar_meta)
        f1s.append(f1)
        epocas.append(epoca)
        if apuntar:
            preds.append(predecir(red, "test", usar_meta).argmax(1).cpu().numpy())
            if s == semillas[0]:
                HISTORIAS[(tarea, nombre)], REDES[(tarea, nombre)] = historia, red.cpu()
        del red
    fila = {"tarea": tarea, "modelo": nombre, "arquitectura": arq, "embedding": emb, "meta": usar_meta, **hp,
            "f1_val": float(np.mean(f1s)), "f1_val_std": float(np.std(f1s)),
            "mejor_epoca": float(np.mean(epocas)), "segundos": time.time() - t0}
    if apuntar:
        detalle = ", ".join(f"{k}={v}" for k, v in hp.items()) + f"; mejor época ≈ {np.mean(epocas):.0f}"
        r = registrar(tarea, "Deep", nombre, preds, f1s, detalle)
        fila.update(f1_test=r["f1"], f1_test_std=r["f1_std"], acc_test=r["accuracy"])
        DEEP_LOG.append(fila)
        print(f"{nombre:55s} F1 val {fila['f1_val']:.3f} ± {fila['f1_val_std']:.3f} | "
              f"F1 test {r['f1']:.3f} ± {r['f1_std']:.3f} | época {np.mean(epocas):.0f} | {fila['segundos']:.0f} s")
    return fila

### 5.5 Búsqueda de hiperparámetros de las redes

Igual que con los modelos clásicos, una rejilla evaluada en validación. Para que la búsqueda no
se mezcle con la comparación de embeddings, se hace con una única variante (Google News
congelado) y una semilla, y los valores elegidos se mantienen fijos en el resto de experimentos.

| Red | Hiperparámetro | Valores |
|---|---|---|
| CNN | filtros por tamaño de ventana | 100, 200 |
| CNN | dropout | 0,3 y 0,5 |
| CNN | pesos por clase en la función de pérdida | no / sí (`balanced`) |
| RNN | tipo de celda | LSTM, GRU |
| RNN | tamaño del estado oculto (por dirección) | 64, 128 |

La RNN tarda bastante más en entrenar, así que su rejilla es más pequeña: hereda de la CNN el
dropout y la decisión sobre los pesos por clase, que dependen sobre todo del desbalanceo de los
datos y no de la arquitectura. La tasa de aprendizaje, el tamaño de lote y los tamaños de
ventana (3, 4, 5) se dejan en valores habituales.

In [ ]:
def buscar(arq, grid):
    """Entrena cada combinación de la rejilla (1 semilla, Google News congelado) y elige por F1 de validación."""
    filas = []
    for hp in (grid[:2] if FAST else grid):
        fila = experimento("queue", arq, "Google News congelado", hp, semillas=SEEDS[:1], apuntar=False)
        filas.append(fila)
        print(f"{arq}: {hp} -> F1 val {fila['f1_val']:.3f} (época {fila['mejor_epoca']:.0f}, {fila['segundos']:.0f} s)")
    filas.sort(key=lambda f: -f["f1_val"])
    display(pd.DataFrame(filas)[[*grid[0], "f1_val", "mejor_epoca", "segundos"]].round(3))
    return {k: filas[0][k] for k in grid[0]}, filas


t0 = time.time()
HP, BUSQUEDA_DEEP = {}, {}
HP["CNN"], BUSQUEDA_DEEP["CNN"] = buscar("CNN", [{"n_filtros": f, "dropout": d, "pesos_clase": w}
                                                 for f in (100, 200) for d in (0.3, 0.5) for w in (False, True)])
HP["RNN"], BUSQUEDA_DEEP["RNN"] = buscar("RNN", [{"celda": c, "oculto": h, "dropout": HP["CNN"]["dropout"],
                                                  "pesos_clase": HP["CNN"]["pesos_clase"], "atencion": "producto escalar"}
                                                 for c in ("LSTM", "GRU") for h in (64, 128)])
ARQS = ["CNN", "RNN"]
NOMBRE_ARQ = {arq: nombre_arq(arq, HP[arq]) for arq in ARQS}
print(f"\nBúsqueda completa en {(time.time() - t0) / 60:.1f} min")
for arq in ARQS:
    print(f"  {NOMBRE_ARQ[arq]}: {HP[arq]}")

### 5.6 Comparación de embeddings

Con los hiperparámetros fijados, cada arquitectura se entrena con las cinco variantes de
embedding y tres semillas. Es el experimento central de la entrega.

In [ ]:
for arq in ARQS:
    for emb in EMBEDDINGS:
        experimento("queue", arq, emb, HP[arq])

COMP = pd.DataFrame([r for r in DEEP_LOG if r["tarea"] == "queue" and not r["meta"]
                     and r["modelo"] == f"{NOMBRE_ARQ[r['arquitectura']]} · {r['embedding']}"])
COMP["red"] = COMP["arquitectura"].map(NOMBRE_ARQ)
display(COMP.pivot(index="embedding", columns="red", values="f1_test").loc[list(EMBEDDINGS)].round(3))

In [ ]:
mejor_shallow_fila = max((r for r in RESULTADOS if r["tarea"] == "queue" and r["familia"] == "Shallow"),
                         key=lambda r: r["f1_val"])
fig, ax = plt.subplots(figsize=(9, 4.6))
alto, y = 0.34, np.arange(len(EMBEDDINGS))
for k, (arq, color) in enumerate(zip(ARQS, SERIE)):
    sub = COMP[COMP["arquitectura"] == arq].set_index("embedding").loc[list(EMBEDDINGS)]
    pos = y + (k - 0.5) * (alto + 0.04)
    ax.barh(pos, sub["f1_test"], height=alto, color=color, label=NOMBRE_ARQ[arq], xerr=sub["f1_test_std"],
            error_kw={"ecolor": COL["tinta2"], "elinewidth": 1, "capsize": 2})
    for p, v, e in zip(pos, sub["f1_test"], sub["f1_test_std"]):
        ax.text(v + e + 0.006, p, f"{v:.3f}", va="center", fontsize=9)
ax.axvline(mejor_shallow_fila["f1"], color=COL["tinta2"], lw=1)
ax.text(mejor_shallow_fila["f1"], -0.72, f"mejor shallow ({mejor_shallow_fila['f1']:.3f})",
        ha="center", va="bottom", fontsize=9, color=COL["tinta2"])
ax.set_yticks(y, list(EMBEDDINGS))
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.tick_params(length=0)
ax.set_xlim(0, max(COMP["f1_test"].max(), mejor_shallow_fila["f1"]) * 1.15)
ax.set_xlabel("F1 macro en test (media de las semillas; la línea fina es la desviación típica)")
ax.set_title("Cola: efecto del embedding en cada arquitectura", pad=18)
ax.legend(loc="lower right")
guardar(fig, "03_comparacion_embeddings")

In [ ]:
# Curvas de aprendizaje de la CNN: qué pasa época a época según el embedding.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for emb, color in zip(["Aleatorio", "Google News congelado", "Google News fine-tuning"], SERIE):
    h = HISTORIAS[("queue", f"CNN · {emb}")]
    for ax, col in zip(axes, ("loss_train", "f1_val")):
        ax.plot(h["epoca"], h[col], color=color, lw=2, label=emb)
axes[0].set_title("CNN: pérdida en train")
axes[1].set_title("CNN: F1 macro en validación")
for ax in axes:
    ax.set_xlabel("época")
axes[1].legend(loc="lower right")
fig.tight_layout()
guardar(fig, "04_curvas_aprendizaje")

### 5.7 ¿Qué aporta la atención?

Se entrena la misma RNN, con su mejor embedding, de tres formas: sin atención (solo el estado
final), con la atención de producto escalar vista en clase y con la variante general. Además
del F1 se mide **dónde cae la atención**: qué parte del peso total se llevan el primer y el
último token del ticket.

In [ ]:
MEJOR_EMB = {arq: COMP[COMP["arquitectura"] == arq].sort_values("f1_val", ascending=False).iloc[0]["embedding"]
             for arq in ARQS}
print("Mejor embedding de cada arquitectura (según validación):", MEJOR_EMB)

emb_rnn = MEJOR_EMB["RNN"]
VARIANTES_ATT = {att: f"{nombre_arq('RNN', {**HP['RNN'], 'atencion': att})} · {emb_rnn}"
                 for att in ("no", "producto escalar", "general")}
for att in ("no", "general"):
    experimento("queue", "RNN", emb_rnn, {**HP["RNN"], "atencion": att})


def atencion_en_test(nombre):
    """Predicción y pesos de atención sobre test de una red ya entrenada (la de la primera semilla)."""
    red = REDES[("queue", nombre)].to(DEVICE)
    logits, alfa = predecir(red, "test", con_atencion=True)
    red.cpu()
    return logits.argmax(1).cpu().numpy(), alfa.cpu().numpy()


L_test = DATOS["test"]["L"].numpy()
filas_test = np.arange(len(L_test))
ATT, filas = {}, []
for att, nombre in VARIANTES_ATT.items():
    r = next(r for r in RESULTADOS if r["tarea"] == "queue" and r["modelo"] == nombre)
    fila = {"variante": nombre_arq("RNN", {**HP["RNN"], "atencion": att}), "accuracy": r["accuracy"],
            "f1": r["f1"], "f1_std": r["f1_std"]}
    if att != "no":
        ATT[att] = atencion_en_test(nombre)
        alfa = ATT[att][1]
        extremos = alfa[filas_test, 0] + np.where(L_test > 1, alfa[filas_test, L_test - 1], 0)
        fila["atención en primer y último token"] = float(extremos.mean())
        fila["peso del token más atendido"] = float(alfa.max(axis=1).mean())
    filas.append(fila)
ATENCION_TABLA = pd.DataFrame(filas)
display(ATENCION_TABLA.round(3))

In [ ]:
import html as _html

# Palabras que más atención reciben en los tickets asignados a cada cola, con cada variante.
# El valor que se promedia es el peso relativo: 1 = lo que tocaría con un reparto uniforme.
MIN_VECES = 3 if FAST else 15


def palabras_atendidas(pred, alfa, k=6):
    suma, veces = {}, {}
    for i, toks in enumerate(TOKENS["test"]):
        n = min(len(toks), MAX_LEN)
        for j in range(n):
            if toks[j].isalpha():
                clave = (pred[i], toks[j])
                suma[clave] = suma.get(clave, 0.0) + alfa[i, j] * n
                veces[clave] = veces.get(clave, 0) + 1
    salida = []
    for c in range(len(TAREAS["queue"])):
        medias = [(w, suma[(cc, w)] / v) for (cc, w), v in veces.items() if cc == c and v >= MIN_VECES]
        salida.append(", ".join(w for w, _ in sorted(medias, key=lambda t: -t[1])[:k]))
    return salida


ATENCION_TOP = pd.DataFrame({"cola": TAREAS["queue"],
                             **{f"atención {att}": palabras_atendidas(*ATT[att]) for att in ATT}})
display(ATENCION_TOP)

In [ ]:
def resaltar(i, att):
    """Ticket de test con cada token sombreado según su peso de atención."""
    pred, alfa = ATT[att]
    toks = TOKENS["test"][i][:MAX_LEN]
    a = alfa[i, :len(toks)] / alfa[i, :len(toks)].max()
    spans = " ".join(f'<span style="background: rgba(42,120,214,{0.6 * x:.2f}); border-radius: 3px; '
                     f'padding: 1px 2px">{_html.escape(t)}</span>' for t, x in zip(toks, a))
    return (f'<p style="line-height: 1.9; color: #0b0b0b; background: #fcfcfb; padding: 8px">'
            f'<b>Atención {att}</b> · real: {TAREAS["queue"][Y["queue"]["test"][i]]} · '
            f'predicho: {TAREAS["queue"][pred[i]]}<br>{spans}</p>')


# Tickets cortos que las dos variantes clasifican bien.
aciertan = np.all([ATT[att][0] == Y["queue"]["test"] for att in ATT], axis=0)
EJEMPLOS_ATT = [int(i) for i in np.flatnonzero(aciertan & (L_test > 20) & (L_test < 60))[:2]]
if not EJEMPLOS_ATT:
    EJEMPLOS_ATT = [int(i) for i in np.flatnonzero(aciertan)[:1]] or [0]
display(HTML("".join(resaltar(i, att) for i in EJEMPLOS_ATT for att in ATT)))

# La misma idea como gráfica: los tokens con más atención de un ticket, con cada variante.
i = EJEMPLOS_ATT[0]
toks = TOKENS["test"][i][:MAX_LEN]
fig, axes = plt.subplots(1, len(ATT), figsize=(11, 3.8))
for ax, att in zip(np.atleast_1d(axes), ATT):
    alfa = ATT[att][1]
    orden = np.argsort(-alfa[i, :len(toks)])[:10]
    barras_h(ax, [f"{toks[j]}  ({j + 1})" for j in orden], [float(alfa[i, j]) for j in orden], fmt="{:.2f}")
    ax.set_title(f"Atención {att}")
    ax.set_xlabel("peso de atención (entre paréntesis, la posición del token)")
fig.suptitle(f"Tokens con más atención en un ticket de «{TAREAS['queue'][Y['queue']['test'][i]]}» "
             f"de {len(toks)} tokens", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout()
guardar(fig, "05_atencion_ejemplo")
ATENCION_EJEMPLO = {"texto": " ".join(toks), "cola": TAREAS["queue"][Y["queue"]["test"][i]], "tokens": len(toks)}

### 5.8 Red híbrida: texto + meta-features

La misma idea que en 4.2, ahora dentro de la red: las 12 meta-features pasan por una capa
pequeña y se concatenan a la representación del texto antes de la capa de salida. Lo probamos
con la mejor configuración de cada arquitectura.

In [ ]:
for arq in ARQS:
    experimento("queue", arq, MEJOR_EMB[arq], HP[arq], usar_meta=True)
display(tabla("queue").round(3))

## 6. Las otras dos salidas: prioridad y tipo

Repetimos el proceso para las otras dos etiquetas de E1. Los modelos clásicos repiten su
búsqueda en rejilla completa (es barata). Las redes reutilizan los hiperparámetros y el mejor
embedding encontrados para la cola; optimizarlos para cada tarea queda para la siguiente
iteración.

In [ ]:
for tarea in ("priority", "type"):
    t0 = time.time()
    GRID_SHALLOW[tarea] = busqueda_shallow(tarea)
    MEJOR_SHALLOW[tarea] = evaluar_shallow(tarea, GRID_SHALLOW[tarea])
    print(f"\n== {NOMBRE_TAREA[tarea]}: grid shallow en {time.time() - t0:.0f} s")
    for arq in ARQS:
        experimento(tarea, arq, MEJOR_EMB[arq], HP[arq])

print("\nPrioridad")
display(tabla("priority", extra=["recall_high"]).round(3))
print("Tipo")
display(tabla("type").round(3))

## 7. Tabla comparativa final y análisis de errores

In [ ]:
FINAL = pd.DataFrame(RESULTADOS)
for tarea in TAREAS:
    print(f"\n=== {NOMBRE_TAREA[tarea]} · métricas en test ===")
    display(tabla(tarea, extra=["recall_high"] if tarea == "priority" else []).round(3))

In [ ]:
# Ranking de todos los modelos de la cola. El color indica la familia.
t = FINAL[FINAL["tarea"] == "queue"].sort_values("f1", ascending=False)
color_familia = {"Shallow": COL["azul"], "Deep": COL["naranja"], "Baseline": COL["gris"]}
fig, ax = plt.subplots(figsize=(9.5, 0.34 * len(t) + 1.2))
barras_h(ax, t["modelo"].tolist(), t["f1"].tolist(), color=[color_familia[f] for f in t["familia"]],
         err=t["f1_std"].tolist())
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, label=f) for f, c in color_familia.items()],
          loc="lower right")
ax.set_xlabel("F1 macro en test")
ax.set_title("Cola: todos los modelos")
guardar(fig, "06_ranking_cola")

In [ ]:
# Matriz de confusión del mejor modelo de la cola (elegido por validación, evaluado en test).
mejor_fila = max((r for r in RESULTADOS if r["tarea"] == "queue"), key=lambda r: r["f1_val"])
MEJOR_MODELO = mejor_fila["modelo"]
y_real, y_pred = Y["queue"]["test"], PREDICCIONES[("queue", MEJOR_MODELO)]
clases = TAREAS["queue"]
cm = confusion_matrix(y_real, y_pred, labels=range(len(clases)), normalize="true")
fig, ax = plt.subplots(figsize=(8.2, 6.6))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(clases)), clases, rotation=40, ha="right")
ax.set_yticks(range(len(clases)), clases)
ax.grid(False)
ax.tick_params(length=0)
for a in range(len(clases)):
    for b in range(len(clases)):
        if cm[a, b] >= 0.005:
            ax.text(b, a, f"{cm[a, b]:.2f}", ha="center", va="center", fontsize=8,
                    color="#ffffff" if cm[a, b] > 0.55 else COL["tinta"])
ax.set_xlabel("cola predicha")
ax.set_ylabel("cola real")
ax.set_title(f"Matriz de confusión (cada fila suma 1) · {MEJOR_MODELO}")
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02).outline.set_visible(False)
guardar(fig, "07_matriz_confusion")

INFORME_CLASES = pd.DataFrame(classification_report(y_real, y_pred, labels=range(len(clases)), target_names=clases,
                                                    output_dict=True, zero_division=0)).T
display(INFORME_CLASES.round(3))

# Las confusiones más frecuentes: qué cola real se confunde con cuál.
errores = pd.DataFrame(confusion_matrix(y_real, y_pred, labels=range(len(clases))), index=clases, columns=clases)
CONFUSIONES = pd.DataFrame([{"real": a, "predicha": b, "tickets": int(errores.loc[a, b])}
                            for a in clases for b in clases if a != b])
CONFUSIONES = CONFUSIONES.sort_values("tickets", ascending=False).head(8).reset_index(drop=True)
display(CONFUSIONES)

In [ ]:
# Se guarda todo lo necesario para el informe.
def limpio(o):
    if isinstance(o, dict):
        return {str(k): limpio(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [limpio(v) for v in o]
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        o = o.item()
    if isinstance(o, float) and np.isnan(o):
        return None
    return o


resumen = {
    "particiones": {s: int(len(i)) for s, i in IDX.items()},
    "tickets_e2": int(len(df_e2)), "tickets_unicos": int(df_e2["clave"].nunique()),
    "fuga": FUGA.to_dict("records"),
    "clases": {t: df[t].value_counts().to_dict() for t in TAREAS},
    "vocabulario": len(vocab), "tfidf_features": {k: int(v["train"].shape[1]) for k, v in TFIDF.items()},
    "mejor_shallow": {t: {n: {"tfidf": m["tfidf"], "params": m["params"], "f1_val": m["f1_val"]}
                          for n, m in d.items()} for t, d in MEJOR_SHALLOW.items()},
    "grid_shallow_queue": GRID_SHALLOW["queue"].to_dict("records"),
    "cobertura": COBERTURA.to_dict("records"),
    "busqueda_deep": BUSQUEDA_DEEP, "hp_deep": HP, "nombre_arq": NOMBRE_ARQ, "mejor_embedding": MEJOR_EMB,
    "deep_log": DEEP_LOG, "resultados": RESULTADOS, "mejor_modelo_cola": MEJOR_MODELO,
    "informe_clases": INFORME_CLASES.reset_index(names="clase").to_dict("records"),
    "confusiones": CONFUSIONES.to_dict("records"),
    "atencion_tabla": ATENCION_TABLA.to_dict("records"), "atencion_top": ATENCION_TOP.to_dict("records"),
    "atencion_ejemplo": ATENCION_EJEMPLO,
    "meta_features": list(META.columns), "semillas": SEEDS, "max_epocas": MAX_EPOCHS, "paciencia": PACIENCIA,
    "lr": LR, "tam_lote": TAM_LOTE, "fast": FAST,
}
(ART_DIR / "resultados.json").write_text(json.dumps(limpio(resumen), ensure_ascii=False, indent=1), encoding="utf-8")
FINAL.to_csv(ART_DIR / "resultados.csv", index=False)
print("Guardado en", ART_DIR / "resultados.json")